In [1]:
import pandas as pd

GOLD_DATA_PATH = "/lakehouse/default/Files/gold_seguranca_viaria"

sinistros = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.silver_infosiga_sinistros").toPandas()
pessoas = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.silver_infosiga_pessoas").toPandas()

StatementMeta(, 02269c1a-3d2b-4efa-99c1-2383e15e5ba2, 3, Finished, Available, Finished, False)

In [2]:
sinistros_oz = sinistros.query(
    "municipio.str.contains('OSASCO', na=False, case=False)"
).copy()

sinistros_oz["data_sinistro"] = pd.to_datetime(
    sinistros_oz["data_sinistro"], dayfirst=True
)

sinistros_oz["ano_mes_sinistro"] = pd.to_datetime(
    sinistros_oz["ano_mes_sinistro"], format="%Y/%m"
)

sinistros_ano_mes_tipo = (
    sinistros_oz.query("ano_mes_sinistro >= '2019-01-01'")
    .copy()
    .groupby(["ano_mes_sinistro", "tipo_registro"], as_index=False)
    .size()
)

sinistros_ano_mes_tipo_via = (
    sinistros_oz.query("ano_mes_sinistro >= '2019-01-01'")
    .copy()
    .groupby(["ano_mes_sinistro", "tipo_via"], as_index=False)
    .size()
)

sinistros_oz["dia_da_semana"] = sinistros_oz["dia_da_semana"].str.upper()
sinistros_oz["turno"] = sinistros_oz["turno"].str.upper()

dias_order = [
    "DOMINGO",
    "SEGUNDA-FEIRA",
    "TERÇA-FEIRA",
    "QUARTA-FEIRA",
    "QUINTA-FEIRA",
    "SEXTA-FEIRA",
    "SÁBADO",
]
turno_order = ["MADRUGADA", "MANHA", "TARDE", "NOITE"]

sinistros_oz["dia_da_semana"] = pd.Categorical(
    sinistros_oz["dia_da_semana"], categories=dias_order, ordered=True
)
sinistros_oz["turno"] = pd.Categorical(
    sinistros_oz["turno"], categories=turno_order, ordered=True
)

sinistros_diasemana_turno = (
    sinistros_oz.query("ano_mes_sinistro >= '2019-01-01'")
    .groupby(["ano_sinistro", "dia_da_semana", "turno"], as_index=False, observed=True)
    .size()
    .query("turno != 'NAO DISPONIVEL'")
)


sinistros_ano_mes_tipo_via.to_parquet(
    GOLD_DATA_PATH + "/gold_infosiga_sinistros_tipo_via.parquet", index=False
)

sinistros_ano_mes_tipo.to_parquet(
    GOLD_DATA_PATH + "/gold_infosiga_sinistros_tipo_registro.parquet", index=False
)

sinistros_diasemana_turno.to_parquet(
    GOLD_DATA_PATH + "/gold_infosiga_sinistros_dia_semana_turno.parquet", index=False
)


StatementMeta(, 02269c1a-3d2b-4efa-99c1-2383e15e5ba2, 4, Finished, Available, Finished, False)

In [3]:
# adicionar filtro por ano
pessoas_oz = pessoas.loc[
    pessoas["municipio"].str.contains("OSASCO", na=False, case=False)
    & (pessoas["ano_sinistro"] >= 2019)
].copy()

pessoas_oz = pessoas_oz[
    [
        "ano_sinistro",
        "id_sinistro",
        "tipo_veiculo_vitima",
        "faixa_etaria_demografica",
        "sexo",
        "tipo_de_vitima",
        "gravidade_lesao",
    ]
].copy()

pessoas_oz.to_parquet(
    GOLD_DATA_PATH + "/gold_infosiga_pessoas_oz.parquet", index=False
)

StatementMeta(, 02269c1a-3d2b-4efa-99c1-2383e15e5ba2, 5, Finished, Available, Finished, False)

In [4]:
sdf_gold_infosiga_sinistros_tipo_via = spark.createDataFrame(sinistros_ano_mes_tipo_via)
(
    sdf_gold_infosiga_sinistros_tipo_via
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_infosiga_sinistros_tipo_via")
)

sdf_gold_infosiga_sinistros_tipo_registro = spark.createDataFrame(sinistros_ano_mes_tipo)
(
    sdf_gold_infosiga_sinistros_tipo_registro
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_infosiga_sinistros_tipo_registro")
)

sdf_gold_infosiga_sinistros_dia_semana_turno = spark.createDataFrame(sinistros_diasemana_turno)
(
    sdf_gold_infosiga_sinistros_dia_semana_turno
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_infosiga_sinistros_dia_semana_turno")
)

sdf_gold_infosiga_pessoas_oz = spark.createDataFrame(pessoas_oz)
(
    sdf_gold_infosiga_pessoas_oz
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_infosiga_pessoas_oz")
)

StatementMeta(, 02269c1a-3d2b-4efa-99c1-2383e15e5ba2, 6, Finished, Available, Finished, False)